In [1]:
import re
import yaml

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

In [2]:
output_path = Path(validation_path) / "output"

input_data = yaml.safe_load((Path(VALIDATION_RUN_INPUTS_DIR)/"input_validation.yml").read_text(encoding="utf-8"))

population_scale_from_input_yml = input_data["population_demographic"]["artificial_rescaling_of_population_size"]
starting_date = input_data["simulation_timeframe"]["starting_date"]
ending_date = input_data["simulation_timeframe"]["ending_date"]

info(f"Checking outputs for validation run: {validation_path}\n")
print(f"Population scale from input yml: {population_scale_from_input_yml}")
print(f"Starting date: {starting_date}")
print(f"Ending date: {ending_date}")

→ Checking outputs for validation run: validation_runs/validation_init_pop_21.12M.asc_0.25_population_scale_one_pattern_20_replicates

Population scale from input yml: 0.25
Starting date: 2011/1/1
Ending date: 2024/1/1


In [3]:
output_1_db = output_path / "validation_monthly_data_3.db"
print(f"Reading monthly data from: {output_1_db}")
# date = get_table_with_columns(output_1_db, ["*"], "monthly_data")
date = get_table(output_1_db, "monthly_data")
# Load yaml input file
date['date'] = pd.to_datetime(starting_date) + pd.to_timedelta(date['days_elapsed'], unit='D')
date = date.rename(columns={"id": "monthly_data_id"})
date = date.sort_values("date")
date_dict = date.set_index("monthly_data_id")["date"].to_dict()

Reading monthly data from: validation_runs/validation_init_pop_21.12M.asc_0.25_population_scale_one_pattern_20_replicates/output/validation_monthly_data_3.db


In [4]:
# Check the range of the date data from the db
print(f"Date range from db: {min(date_dict.values())} to {max(date_dict.values())}")

print(f"Date range from input yml: {starting_date} to {ending_date}")


db_min = min(date_dict.values())
db_max = max(date_dict.values())
expected_start = pd.to_datetime(starting_date)
expected_end = pd.to_datetime(ending_date)

mismatches = []
if db_min != expected_start:
    mismatches.append(
        f"STARTING DATE mismatch: db min = {db_min}, yml starting_date = {expected_start}"
    )
if db_max != expected_end:
    mismatches.append(
        f"ENDING DATE mismatch: db max = {db_max}, yml ending_date = {expected_end}"
    )

if mismatches:
    error(mismatches)
    raise ValueError(
        "Date range from db does not match date range from input yml:\n"
        + "\n".join(mismatches)
    )
else:
    ok("Date range from db matches date range from input yml.")

Date range from db: 2011-01-01 00:00:00 to 2024-01-01 00:00:00
Date range from input yml: 2011/1/1 to 2024/1/1
✓ Date range from db matches date range from input yml.


In [5]:
log_path = Path(validation_path) / "log"

for log_file in log_path.glob("*.log"):
    with open(log_file, "r") as f:
        log_content = f.read()

        # print the last line of the log file
        last_line = log_content.strip().split("\n")[-1]
        print(f"Last line of log file {log_file.name}: {last_line}")
    

Last line of log file validation_rep_15.log: Memory Usage: 10.16 GB
Last line of log file validation_rep_6.log: Memory Usage: 10.17 GB
Last line of log file validation_rep_19.log: Memory Usage: 10.18 GB
Last line of log file validation_rep_16.log: Memory Usage: 10.17 GB
Last line of log file validation_rep_18.log: Memory Usage: 10.18 GB
Last line of log file validation_rep_3.log: Memory Usage: 10.18 GB
Last line of log file validation_rep_13.log: Memory Usage: 10.16 GB
Last line of log file validation_rep_1.log: Memory Usage: 10.17 GB
Last line of log file validation_rep_9.log: Memory Usage: 10.20 GB
Last line of log file validation_rep_11.log: Memory Usage: 10.21 GB
Last line of log file validation_rep_10.log: Memory Usage: 10.17 GB
Last line of log file validation_rep_17.log: Memory Usage: 10.17 GB
Last line of log file validation_rep_14.log: Memory Usage: 10.19 GB
Last line of log file validation_rep_2.log: Memory Usage: 10.19 GB
Last line of log file validation_rep_20.log: Memory U

In [6]:
# # find last day in log file in format [2026-09-03 17:29:02] [info] Day: 120
# current_days = []
# current_timestamps = []
# for log_file in log_path.glob("*.log"):
#     with open(log_file, "r") as f:
#         log_content = f.read()

#         # find all lines that contain "Day: "
#         day_lines = [line for line in log_content.strip().split("\n") if "Day: " in line]

#         if day_lines:
#             last_day_line = day_lines[-1]
#             last_day = int(last_day_line.split("Day: ")[-1])
#             last_timestamp_line = day_lines[-1]
#             last_timestamp = last_timestamp_line.split("]")[0][1:]
#             current_timestamps.append(last_timestamp)
#             current_days.append(last_day)
#         else:
#             print(f"No 'Day: ' lines found in log file {log_file.name}")

# total_days = (pd.to_datetime(ending_date) - pd.to_datetime(starting_date)).days

# print(f"Total days in simulation: ~{total_days}")

# average_current_day = sum(current_days) / len(current_days) if current_days else 0
# percent_complete = (average_current_day / total_days) * 100 if total_days > 0 else 0
# print(f"Average current day across log files: {average_current_day:.2f}")

# # Color thresholds: Red (<50%), Yellow (<80%), Green (80%+)
# if percent_complete < 50:
#     color_code = "\033[91m"  # Red
# elif percent_complete < 80:
#     color_code = "\033[93m"  # Yellow
# else:
#     color_code = "\033[92m"  # Green

# reset_code = "\033[0m"

# print(f"Progress estimate = {color_code}{percent_complete:.2f}%{reset_code}")

# # get first timestamp in log files
# first_timestamps = []
# for log_file in log_path.glob("*.log"):
#     with open(log_file, "r") as f:
#         log_content = f.read()

#         # find all lines that contain a timestamp in the format [YYYY-MM-DD HH:MM:SS]
#         timestamp_lines = [
#             line for line in log_content.strip().split("\n") if line.startswith("[")
#         ]

#         if timestamp_lines:
#             first_timestamp_line = timestamp_lines[0]
#             first_timestamp = first_timestamp_line.split("]")[0][1:]
#             first_timestamps.append(first_timestamp)
#         else:
#             print(f"No timestamp lines found in log file {log_file.name}")

# # using the last timestamps, first timestamps, and the percent complete, estimate the expected end date of the simulation
# if current_timestamps and first_timestamps:
#     last_timestamp = max(current_timestamps)
#     first_timestamp = min(first_timestamps)

#     # Calculate the elapsed time in seconds
#     elapsed_time = (pd.to_datetime(last_timestamp) - pd.to_datetime(first_timestamp)).total_seconds()

#     # Estimate total time based on percent complete
#     estimated_total_time = elapsed_time / (percent_complete / 100) if percent_complete > 0 else 0

#     # Calculate the expected end timestamp
#     expected_end_timestamp = pd.to_datetime(first_timestamp) + pd.to_timedelta(estimated_total_time, unit='s')

#     # Calculate time remaining in hours, minutes, and seconds
#     time_remaining = expected_end_timestamp - pd.to_datetime(last_timestamp)
#     hours, remainder = divmod(time_remaining.total_seconds(), 3600)
#     minutes, seconds = divmod(remainder, 60)

#     print(f"Current timestamp: {last_timestamp}")
#     print(f"Estimated end timestamp of simulation: {expected_end_timestamp:%Y-%m-%d %H:%M:%S}")
#     print(f"Time remaining: {int(hours)} hours, {int(minutes)} minutes, {int(seconds)} seconds")

In [9]:
DAY_LINE_RE = re.compile(
    r"^\[(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?\bDay:\s*(?P<day>\d+)\b"
)

records = []  # one record per log file with first/last timestamp+day

for log_file in log_path.glob("*.log"):
    parsed = []

    with open(log_file, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            m = DAY_LINE_RE.match(line.strip())
            if m:
                ts = pd.to_datetime(m.group("ts"), errors="coerce")
                day = int(m.group("day"))
                if pd.notna(ts):
                    parsed.append((ts, day))

    if not parsed:
        print(f"No parseable 'Day:' lines found in {log_file.name}")
        continue

    # parsed is already in file order; enforce chronological sort just in case
    parsed.sort(key=lambda x: x[0])

    first_ts, first_day = parsed[0]
    last_ts, last_day = parsed[-1]

    elapsed_seconds = (last_ts - first_ts).total_seconds()
    day_progress = last_day - first_day

    # speed in "simulation days per second"
    speed_days_per_sec = (day_progress / elapsed_seconds) if elapsed_seconds > 0 else np.nan

    records.append(
        {
            "file": log_file.name,
            "first_ts": first_ts,
            "last_ts": last_ts,
            "first_day": first_day,
            "last_day": last_day,
            "elapsed_seconds": elapsed_seconds,
            "day_progress": day_progress,
            "speed_days_per_sec": speed_days_per_sec,
        }
    )

if not records:
    raise RuntimeError("No usable log files found.")

df = pd.DataFrame(records)

# --- Progress calculation ---
start_dt = pd.to_datetime(starting_date)
end_dt = pd.to_datetime(ending_date)

# If your simulation is inclusive of both endpoints, use +1:
# total_days = (end_dt - start_dt).days + 1
total_days = round((end_dt - start_dt).days / 30) * 30


if total_days <= 0:
    raise ValueError("ending_date must be after starting_date (or adjust inclusive logic).")

average_current_day = df["last_day"].mean()
percent_complete = (average_current_day / total_days) * 100

print(f"Total days in simulation: ~{total_days}")
print(f"Average current day across log files: {average_current_day:.2f}")

# Color thresholds: Red (<50%), Yellow (<80%), Green (80%+)
if percent_complete < 50:
    color_code = "\033[91m"  # Red
elif percent_complete < 80:
    color_code = "\033[93m"  # Yellow
else:
    color_code = "\033[92m"  # Green
reset_code = "\033[0m"

print(f"Progress estimate = {color_code}{percent_complete:.2f}%{reset_code}")

# --- ETA calculation (robust) ---
# Use median speed across files to reduce outlier impact
valid_speeds = df["speed_days_per_sec"].replace([np.inf, -np.inf], np.nan).dropna()
valid_speeds = valid_speeds[valid_speeds > 0]

if len(valid_speeds) == 0:
    print("Not enough data to estimate ETA (no positive speed measurements).")
else:
    median_speed = valid_speeds.median()  # days/sec
    remaining_days = max(total_days - average_current_day, 0)

    remaining_seconds = remaining_days / median_speed
    remaining_td = pd.to_timedelta(remaining_seconds, unit="s")

    # "Now" = latest observed timestamp among files
    current_timestamp = df["last_ts"].max()
    expected_end_timestamp = current_timestamp + remaining_td

    hrs, rem = divmod(int(remaining_td.total_seconds()), 3600)
    mins, secs = divmod(rem, 60)

    print(f"Current timestamp: {current_timestamp:%Y-%m-%d %H:%M:%S}")
    print(f"Estimated end timestamp of simulation: {expected_end_timestamp:%Y-%m-%d %H:%M:%S}")
    print(f"Time remaining: {hrs} hours, {mins} minutes, {secs} seconds")


Total days in simulation: ~4740
Average current day across log files: 4740.00
Progress estimate = 100.00%
Current timestamp: 2026-09-05 01:22:04
Estimated end timestamp of simulation: 2026-09-05 01:22:04
Time remaining: 0 hours, 0 minutes, 0 seconds
